## Daphnia paraliż 6000 

Analiza koncowych 20 sekund nagran `ruch` w ROI x=150..850 px i na calej wysokosci klatki. Skala fizyczna jest orientacyjna i wymaga pozniejszej kalibracji.

In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

replicate_name = "20250909"
root_dir = Path("/media/marcelina/marcelina/D. magna") / replicate_name
output_dir = Path("/home/marcelina/D.magna/Krótki") / replicate_name
output_dir.mkdir(parents=True, exist_ok=True)

index_path = output_dir / "all_recordings_index.csv"
if not index_path.exists():
    raise FileNotFoundError(f"Brak indeksu nagran: {index_path}")
conditions_df = pd.read_csv(index_path)
heart_folder = root_dir

roi_path = output_dir / "roi_coordinates_heart.csv"
roi_df = pd.read_csv(roi_path) if roi_path.exists() else pd.DataFrame()
brightness_path = output_dir / "brightness_timecourses.pkl"
brightness_timecourses_df = pd.read_pickle(brightness_path) if brightness_path.exists() else pd.DataFrame()


def wczytaj_koncowe_20_s(video_path, max_seconds=20, grayscale=False):
    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise OSError(f"Nie mozna otworzyc pliku: {video_path}")
    fps = capture.get(cv2.CAP_PROP_FPS)
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if fps <= 0 or frame_count <= 0:
        capture.release()
        raise ValueError(f"Nieprawidlowe parametry pliku: {video_path}")
    frames_to_keep = min(frame_count, int(round(max_seconds * fps)))
    first_frame = max(0, frame_count - frames_to_keep)
    metadata = {
        "path": str(video_path),
        "fps": fps,
        "liczba_klatek": frames_to_keep,
        "szerokosc": int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
        "wysokosc": int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
        "czas_s": frames_to_keep / fps,
        "poczatkowa_klatka": first_frame,
    }

    def frame_generator():
        try:
            capture.set(cv2.CAP_PROP_POS_FRAMES, first_frame)
            for _ in range(frames_to_keep):
                success, frame = capture.read()
                if not success:
                    break
                if grayscale:
                    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
                yield frame
        finally:
            capture.release()

    return metadata, frame_generator()

print(f"Odtworzono indeks: {len(conditions_df)} rekordow")
print(f"Nagrania ruchu: {conditions_df['pomiar'].eq('ruch').sum()}")
print(f"PKL przebiegow jasnosci: {len(brightness_timecourses_df)} rekordow")

In [ ]:
MOTION_ROI_X_START_PX = 150
MOTION_ROI_X_END_PX = 850
MOTION_ASSUMED_WIDTH_MM = 10.0
MOTION_MM_PER_PX = MOTION_ASSUMED_WIDTH_MM / (MOTION_ROI_X_END_PX - MOTION_ROI_X_START_PX)
MOTION_MAX_SECONDS = 20
MOTION_BACKGROUND_SAMPLES = 25
MOTION_MIN_AREA_FRACTION = 0.0005
MOTION_MAX_AREA_FRACTION = 0.35
MOTION_THRESHOLD_OFFSET = 8
MOTION_SMOOTHING_WINDOW = 5


def wyznacz_roi_komory(frame_shape):
    frame_height, frame_width = frame_shape[:2]
    left = MOTION_ROI_X_START_PX
    right = min(MOTION_ROI_X_END_PX, frame_width)
    if not (0 <= left < right <= frame_width):
        raise ValueError("Poziome granice ROI wychodza poza klatke.")
    return left, 0, right, frame_height


def segmentuj_dafnie(frames):
    if len(frames) < 2:
        raise ValueError("Za malo klatek do segmentacji ruchu.")
    left, top, right, bottom = wyznacz_roi_komory(frames[0].shape)
    chamber_frames = [frame[top:bottom, left:right] for frame in frames]
    sample_indices = np.linspace(0, len(chamber_frames) - 1, min(MOTION_BACKGROUND_SAMPLES, len(chamber_frames)), dtype=int)
    background = np.median(np.stack([chamber_frames[i] for i in sample_indices]), axis=0).astype(np.uint8)
    kernel = np.ones((3, 3), dtype=np.uint8)
    min_area = max(1, int(MOTION_MIN_AREA_FRACTION * background.size))
    max_area = int(MOTION_MAX_AREA_FRACTION * background.size)
    rows = []
    for frame_number, chamber_frame in enumerate(chamber_frames):
        difference = cv2.absdiff(chamber_frame, background)
        threshold_value, _ = cv2.threshold(difference, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        threshold_value = max(float(threshold_value), MOTION_THRESHOLD_OFFSET)
        mask = cv2.threshold(difference, threshold_value, 255, cv2.THRESH_BINARY)[1]
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
        count, labels, stats, centroids = cv2.connectedComponentsWithStats(mask, 8)
        candidates = [i for i in range(1, count) if min_area <= stats[i, cv2.CC_STAT_AREA] <= max_area]
        if candidates:
            component = max(candidates, key=lambda i: stats[i, cv2.CC_STAT_AREA])
            area = int(stats[component, cv2.CC_STAT_AREA])
            centroid_x, centroid_y = centroids[component]
            detected = True
        else:
            area, centroid_x, centroid_y, detected = 0, np.nan, np.nan, False
        rows.append({"klatka": frame_number, "x_px": left + centroid_x if detected else np.nan, "y_px": top + centroid_y if detected else np.nan, "pole_maski_px": area, "wykryto": detected, "prog": threshold_value})
    return pd.DataFrame(rows), (left, top, right, bottom), background


def oblicz_parametry_ruchu(detections, fps, chamber):
    result = detections.copy()
    left, top, right, bottom = chamber
    result["czas_s"] = result["klatka"] / fps
    result["x_mm"] = (result["x_px"] - left) * MOTION_MM_PER_PX
    result["y_mm"] = (result["y_px"] - top) * MOTION_MM_PER_PX
    for axis in ("x_mm", "y_mm"):
        result[f"{axis}_wygładzone"] = result[axis].interpolate(limit_direction="both").rolling(MOTION_SMOOTHING_WINDOW, center=True, min_periods=1).median()
    result["droga_klatka_mm"] = np.hypot(result["x_mm_wygładzone"].diff(), result["y_mm_wygładzone"].diff())
    result["droga_skokowo_mm"] = result["droga_klatka_mm"].cumsum().fillna(0)
    result["predkosc_mm_s"] = result["droga_klatka_mm"].fillna(0) * fps
    result["przyspieszenie_mm_s2"] = result["predkosc_mm_s"].diff().fillna(0) * fps
    result["roi_width_mm"] = (right - left) * MOTION_MM_PER_PX
    result["roi_height_mm"] = (bottom - top) * MOTION_MM_PER_PX
    return result

In [ ]:
motion_metadata = conditions_df[conditions_df["path"].notna() & conditions_df["pomiar"].eq("ruch")].drop_duplicates("path").set_index("path")
motion_detection_frames, motion_summary_rows, motion_errors = [], [], []
motion_diagnostics = {}
for video_path, metadata_row in motion_metadata.iterrows():
    try:
        video_info, frame_generator = wczytaj_koncowe_20_s(video_path, max_seconds=MOTION_MAX_SECONDS, grayscale=True)
        frames = list(frame_generator)
        detections, chamber, background = segmentuj_dafnie(frames)
        movement = oblicz_parametry_ruchu(detections, video_info["fps"], chamber)
        movement.insert(0, "file_name", metadata_row["file_name"])
        movement.insert(1, "path", video_path)
        movement["kalibracja"] = "orientacyjna: 10 mm / 700 px"
        for column in ("data", "punkt_czasowy_h", "stężenie_mg", "osobnik"):
            movement[column] = metadata_row[column]
        motion_detection_frames.append(movement)
        motion_summary_rows.append({"file_name": metadata_row["file_name"], "path": video_path, "data": metadata_row["data"], "punkt_czasowy_h": metadata_row["punkt_czasowy_h"], "stężenie_mg": metadata_row["stężenie_mg"], "osobnik": metadata_row["osobnik"], "liczba_klatek": len(movement), "odsetek_wykrytych": movement["wykryto"].mean(), "droga_calkowita_mm": movement["droga_skokowo_mm"].iloc[-1], "srednia_predkosc_mm_s": movement["predkosc_mm_s"].mean(), "maks_predkosc_mm_s": movement["predkosc_mm_s"].max(), "maks_przyspieszenie_mm_s2": movement["przyspieszenie_mm_s2"].abs().max()})
        motion_diagnostics[video_path] = (frames[len(frames) // 2], background, movement, chamber)
    except (KeyError, OSError, ValueError, cv2.error) as error:
        motion_errors.append({"path": video_path, "blad": str(error)})
motion_df = pd.concat(motion_detection_frames, ignore_index=True) if motion_detection_frames else pd.DataFrame()
motion_summary_df = pd.DataFrame(motion_summary_rows)
motion_df.to_csv(output_dir / "motion_detection_checkpoint.csv", index=False, encoding="utf-8-sig")
motion_summary_df.to_csv(output_dir / "motion_summary_checkpoint.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(motion_errors).to_csv(output_dir / "motion_errors.csv", index=False, encoding="utf-8-sig")
print(f"Przeanalizowano nagran: {len(motion_summary_df)} / {len(motion_metadata)}")
display(motion_summary_df)
if motion_errors:
    display(pd.DataFrame(motion_errors))

### Lekka ekstrakcja tabeli zbiorczej (bez rysowania wykresów per nagranie)

Ta komórka powiela wyłącznie logikę budowania tabel wynikowych z ostatniej komórki notebooka
(`motion_recording_summary_df` i `motion_summary_by_concentration_time_df`), pomijając
generowanie diagnostycznych wykresów per nagranie. Pozwala to szybko uzyskać gotowe CSV do
dalszej analizy statystycznej, bez czekania na renderowanie wielu figur.

In [ ]:
# Ekstrakcja tabeli zbiorczej lokomocji do CSV (bez plotowania)
recording_rows_extract = []
for video_path, group in motion_df.groupby("path"):
    metadata = group.iloc[0]
    recording_rows_extract.append({
        "path": video_path,
        "data": metadata["data"],
        "punkt_czasowy_h": metadata["punkt_czasowy_h"],
        "stężenie_mg": metadata["stężenie_mg"],
        "osobnik": metadata["osobnik"],
        "srednia_predkosc_mm_s": group["predkosc_mm_s"].mean(),
        "srednie_abs_przyspieszenie_mm_s2": group["przyspieszenie_mm_s2"].abs().mean(),
        "odsetek_wykrytych": group["wykryto"].mean(),
    })
motion_recording_summary_df = pd.DataFrame(recording_rows_extract)
motion_recording_summary_df.to_csv(output_dir / "motion_recording_summary.csv", index=False, encoding="utf-8-sig")

motion_summary_by_concentration_time_df = (
    motion_recording_summary_df
    .groupby(["data", "punkt_czasowy_h", "stężenie_mg"], as_index=False)
    [["srednia_predkosc_mm_s", "srednie_abs_przyspieszenie_mm_s2"]]
    .mean()
    .sort_values(["punkt_czasowy_h", "stężenie_mg"])
)
motion_summary_by_concentration_time_df.to_csv(
    output_dir / "motion_summary_by_concentration_time.csv", index=False, encoding="utf-8-sig"
)

print(f"Zapisano: {output_dir / 'motion_recording_summary.csv'} ({len(motion_recording_summary_df)} wierszy)")
print(f"Zapisano: {output_dir / 'motion_summary_by_concentration_time.csv'} ({len(motion_summary_by_concentration_time_df)} wierszy)")
display(motion_recording_summary_df)

In [ ]:
from matplotlib.collections import LineCollection


def rysuj_trajektorie_z_predkoscia_final(axis, movement, x_limit, y_limit):
    valid = movement[["x_mm", "y_mm", "predkosc_mm_s"]].dropna().reset_index(drop=True)
    if len(valid) < 2:
        return None
    points = valid[["x_mm", "y_mm"]].to_numpy().reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    speed_max = max(float(valid["predkosc_mm_s"].max()), 1.0)
    collection = LineCollection(segments, cmap="viridis", norm=plt.Normalize(0, speed_max))
    collection.set_array(valid["predkosc_mm_s"].to_numpy()[:-1])
    collection.set_linewidth(2.5)
    axis.add_collection(collection)
    axis.set(xlim=(0, x_limit), ylim=(0, y_limit), xlabel="X [mm]", ylabel="Y [mm]")
    axis.set_aspect("equal")
    return collection


if not motion_df.empty:
    x_limit = float(motion_df["roi_width_mm"].max())
    y_limit = float(motion_df["roi_height_mm"].max())
    example_path = motion_df["path"].iloc[0]
    example_motion = motion_df[motion_df["path"] == example_path]
    metadata = example_motion.iloc[0]

    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    axes = axes.ravel()
    heatmap, _, _ = np.histogram2d(example_motion["x_mm"].dropna(), example_motion["y_mm"].dropna(), bins=(30, 30), range=((0, x_limit), (0, y_limit)))
    axes[0].imshow(heatmap.T, origin="lower", extent=(0, x_limit, 0, y_limit), aspect="auto", cmap="magma")
    axes[0].set(xlabel="X [mm]", ylabel="Y [mm]", title="Heatmapa pozycji")
    collection = rysuj_trajektorie_z_predkoscia_final(axes[1], example_motion, x_limit, y_limit)
    axes[1].set_title("Trajektoria kolorowana prędkością")
    if collection is not None:
        fig.colorbar(collection, ax=axes[1], label="Prędkość [mm/s]")
    axes[2].plot(example_motion["czas_s"], example_motion["predkosc_mm_s"])
    axes[2].set(xlabel="Czas [s]", ylabel="Prędkość [mm/s]", title="Prędkość w czasie")
    axes[3].plot(example_motion["czas_s"], example_motion["przyspieszenie_mm_s2"])
    axes[3].set(xlabel="Czas [s]", ylabel="Przyspieszenie [mm/s²]", title="Przyspieszenie w czasie")
    fig.suptitle(f"{metadata['data']} | {int(metadata['punkt_czasowy_h'])} h | {metadata['stężenie_mg']:g} mg | osobnik {int(metadata['osobnik'])} | {Path(example_path).name}")
    fig.tight_layout()
    fig.savefig(output_dir / "motion_diagnostic_speed_acceleration.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    recording_rows = []
    for video_path, group in motion_df.groupby("path"):
        metadata = group.iloc[0]
        recording_rows.append({
            "path": video_path,
            "data": metadata["data"],
            "punkt_czasowy_h": metadata["punkt_czasowy_h"],
            "stężenie_mg": metadata["stężenie_mg"],
            "osobnik": metadata["osobnik"],
            "srednia_predkosc_mm_s": group["predkosc_mm_s"].mean(),
            "srednie_abs_przyspieszenie_mm_s2": group["przyspieszenie_mm_s2"].abs().mean(),
            "odsetek_wykrytych": group["wykryto"].mean(),
        })
        fig, axes = plt.subplots(2, 2, figsize=(16, 11))
        axes = axes.ravel()
        heatmap, _, _ = np.histogram2d(group["x_mm"].dropna(), group["y_mm"].dropna(), bins=(30, 30), range=((0, x_limit), (0, y_limit)))
        axes[0].imshow(heatmap.T, origin="lower", extent=(0, x_limit, 0, y_limit), aspect="auto", cmap="magma")
        axes[0].set(xlabel="X [mm]", ylabel="Y [mm]", title="Heatmapa pozycji")
        collection = rysuj_trajektorie_z_predkoscia_final(axes[1], group, x_limit, y_limit)
        axes[1].set_title("Trajektoria kolorowana prędkością")
        if collection is not None:
            fig.colorbar(collection, ax=axes[1], label="Prędkość [mm/s]")
        axes[2].plot(group["czas_s"], group["predkosc_mm_s"])
        axes[2].set(xlabel="Czas [s]", ylabel="Prędkość [mm/s]", title="Prędkość w czasie")
        axes[3].plot(group["czas_s"], group["przyspieszenie_mm_s2"])
        axes[3].set(xlabel="Czas [s]", ylabel="Przyspieszenie [mm/s²]", title="Przyspieszenie w czasie")
        for axis in axes:
            axis.grid(True, alpha=0.25)
        fig.suptitle(f"{metadata['data']} | {int(metadata['punkt_czasowy_h'])} h | {metadata['stężenie_mg']:g} mg | osobnik {int(metadata['osobnik'])} | {Path(video_path).name}")
        fig.tight_layout()
        fig.savefig(output_dir / f"motion_analysis_recording_{Path(video_path).stem.replace(' ', '_')}.png", dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    motion_recording_summary_df = pd.DataFrame(recording_rows)
    motion_recording_summary_df.to_csv(output_dir / "motion_recording_summary.csv", index=False, encoding="utf-8-sig")
    motion_summary_by_concentration_time_df = (
        motion_recording_summary_df.groupby(["data", "punkt_czasowy_h", "stężenie_mg"], as_index=False)[["srednia_predkosc_mm_s", "srednie_abs_przyspieszenie_mm_s2"]]
        .mean()
        .sort_values(["punkt_czasowy_h", "stężenie_mg"])
    )
    motion_summary_by_concentration_time_df.to_csv(output_dir / "motion_summary_by_concentration_time.csv", index=False, encoding="utf-8-sig")
    fig, summary_axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)
    for concentration, group in motion_summary_by_concentration_time_df.groupby("stężenie_mg"):
        group = group.sort_values("punkt_czasowy_h")
        label = f"{concentration:g} mg"
        summary_axes[0].plot(group["punkt_czasowy_h"], group["srednia_predkosc_mm_s"], marker="o", label=label)
        summary_axes[1].plot(group["punkt_czasowy_h"], group["srednie_abs_przyspieszenie_mm_s2"], marker="o", label=label)
    summary_axes[0].set(xlabel="Punkt czasowy [h]", ylabel="Średnia prędkość [mm/s]", title="Średnia prędkość")
    summary_axes[1].set(xlabel="Punkt czasowy [h]", ylabel="Średnie |przyspieszenie| [mm/s²]", title="Średnie bezwzględne przyspieszenie")
    for axis in summary_axes:
        axis.set_xticks(sorted(motion_summary_by_concentration_time_df["punkt_czasowy_h"].unique()))
        axis.grid(True, alpha=0.3)
        axis.legend(title="Stężenie")
    fig.suptitle("Ruchliwość Dafni: średnie względem punktu czasowego i stężenia")
    fig.tight_layout()
    fig.savefig(output_dir / "motion_summary_speed_acceleration.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    display(motion_recording_summary_df)
    display(motion_summary_by_concentration_time_df)
else:
    print("Brak wyników ruchu do wizualizacji.")

## 10. Statystyka repeated measures dla aktywności lokomotorycznej

Analogiczny schemat jak dla tempa bicia serca: (a) testy założeń, (b) ścieżka decyzyjna
zależna od wyniku testów założeń (model parametryczny lub transformacja/GLMM),
(c) alternatywa nieparametryczna. Główną zmienną jest `droga_calkowita_mm` (całkowity
dystans pokonany w 20-sekundowym nagraniu) z tabeli `motion_summary_df` — zmień `VALUE_COL`
poniżej na `"srednia_predkosc_mm_s"`, jeśli wolisz analizować średnią prędkość.

In [ ]:
from statsmodels.genmod.families import Gamma, Tweedie
from statsmodels.genmod.families.links import Log
from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.cov_struct import Exchangeable

In [ ]:
VALUE_COL = "droga_calkowita_mm"

motion_summary_df["osobnik_id"] = (
    motion_summary_df["data"].astype(str) + "_"
    + motion_summary_df["stężenie_mg"].map(lambda v: f"{v:g}") + "mg_"
    + motion_summary_df["osobnik"].astype(int).astype(str)
)

In [ ]:
assumptions_model_motion = smf.ols(
    f"{VALUE_COL} ~ C(stężenie_mg) * C(punkt_czasowy_h)", data=motion_summary_df
).fit()
residuals_motion = assumptions_model_motion.resid
shapiro_stat_motion, shapiro_p_motion = scipy_stats.shapiro(residuals_motion)
normality_ok_motion = shapiro_p_motion >= 0.05
print(f"Shapiro-Wilk (reszty modelu): W={shapiro_stat_motion:.4f}, p={shapiro_p_motion:.4g}")
print("Normalność reszt:", "OK" if normality_ok_motion else "ZŁAMANA")

motion_summary_df["_grupa_lev"] = (
    motion_summary_df["stężenie_mg"].astype(str) + "_" + motion_summary_df["punkt_czasowy_h"].astype(str)
)
levene_groups_motion = [g[VALUE_COL].to_numpy() for _, g in motion_summary_df.groupby("_grupa_lev")]
levene_stat_motion, levene_p_motion = scipy_stats.levene(*levene_groups_motion, center="median")
homogeneity_ok_motion = levene_p_motion >= 0.05
print(f"\nLevene: stat={levene_stat_motion:.4f}, p={levene_p_motion:.4g}")
print("Homogeniczność wariancji:", "OK" if homogeneity_ok_motion else "ZŁAMANA")

has_zeros_motion = (motion_summary_df[VALUE_COL] <= 0).any()
print(f"\nCzy zmienna \'{VALUE_COL}\' zawiera zera (całkowicie nieruchome osobniki)?", has_zeros_motion)

assumptions_summary_motion = pd.DataFrame([{
    "test": "Shapiro-Wilk (reszty)", "statystyka": shapiro_stat_motion, "p_value": shapiro_p_motion,
    "wniosek": "normalność OK" if normality_ok_motion else "normalność złamana",
}, {
    "test": "Levene", "statystyka": levene_stat_motion, "p_value": levene_p_motion,
    "wniosek": "homogeniczność OK" if homogeneity_ok_motion else "homogeniczność złamana",
}])
assumptions_summary_motion.to_csv(output_dir / "motion_assumptions_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
if normality_ok_motion and homogeneity_ok_motion:
    print(">>> Założenia spełnione: two-way repeated measures ANOVA (pingouin mixed_anova)")
    mixed_anova_motion = pg.mixed_anova(
        data=motion_summary_df, dv=VALUE_COL, within="punkt_czasowy_h",
        between="stężenie_mg", subject="osobnik_id",
    )
    mixed_anova_motion.to_csv(output_dir / "motion_mixed_anova.csv", index=False, encoding="utf-8-sig")
    display(mixed_anova_motion)

    tukey_motion = pg.pairwise_tukey(data=motion_summary_df, dv=VALUE_COL, between="stężenie_mg")
    tukey_motion.to_csv(output_dir / "motion_tukey_hsd.csv", index=False, encoding="utf-8-sig")
    print("\nTukey HSD:")
    display(tukey_motion)
else:
    print(">>> Założenia złamane: próba transformacji (log1p, sqrt) przed powrotem do modelu parametrycznego")
    transformation_results = []
    for name, transform in [("log1p", np.log1p), ("sqrt", np.sqrt)]:
        transformed = transform(motion_summary_df[VALUE_COL].clip(lower=0))
        t_model = smf.ols(
            "transformed ~ C(stężenie_mg) * C(punkt_czasowy_h)",
            data=motion_summary_df.assign(transformed=transformed),
        ).fit()
        t_stat, t_p = scipy_stats.shapiro(t_model.resid)
        transformation_results.append({"transformacja": name, "shapiro_p": t_p, "normalnosc_po_transformacji": t_p >= 0.05})
        print(f"  Transformacja {name}: Shapiro p={t_p:.4g} ({'OK' if t_p >= 0.05 else 'nadal złamana'})")
    pd.DataFrame(transformation_results).to_csv(output_dir / "motion_transformation_attempts.csv", index=False, encoding="utf-8-sig")

    print("\n>>> Alternatywa: GLMM (tu: GEE jako praktyczne przybliżenie modelu mieszanego)")
    family_motion = Tweedie(link=Log(), var_power=1.5) if has_zeros_motion else Gamma(link=Log())
    print(f"    Wybrany rozkład: {'Tweedie (var_power=1.5)' if has_zeros_motion else 'Gamma'}")

    glmm_df_motion = motion_summary_df.copy()
    if not has_zeros_motion:
        glmm_df_motion = glmm_df_motion[glmm_df_motion[VALUE_COL] > 0]
    else:
        glmm_df_motion[VALUE_COL] = glmm_df_motion[VALUE_COL].clip(lower=0)

    gee_model_motion = GEE.from_formula(
        f"{VALUE_COL} ~ C(stężenie_mg) * punkt_czasowy_h",
        groups="osobnik_id", data=glmm_df_motion,
        cov_struct=Exchangeable(), family=family_motion,
    )
    gee_fit_motion = gee_model_motion.fit()
    print(gee_fit_motion.summary())
    with open(output_dir / "motion_gee_report.txt", "w", encoding="utf-8") as report_file:
        report_file.write(gee_fit_motion.summary().as_text())

In [ ]:
kw_rows_motion = []
dunn_frames_motion = []
for time_point, group in motion_summary_df.groupby("punkt_czasowy_h"):
    samples = [g[VALUE_COL].to_numpy() for _, g in group.groupby("stężenie_mg")]
    if len(samples) >= 2:
        kw_stat, kw_p = scipy_stats.kruskal(*samples)
        kw_rows_motion.append({"punkt_czasowy_h": time_point, "H": kw_stat, "p_value": kw_p})
        dunn_result = sp.posthoc_dunn(group, val_col=VALUE_COL, group_col="stężenie_mg", p_adjust="holm")
        dunn_result["punkt_czasowy_h"] = time_point
        dunn_frames_motion.append(dunn_result)

kw_df_motion = pd.DataFrame(kw_rows_motion)
kw_df_motion.to_csv(output_dir / "motion_kruskal_wallis.csv", index=False, encoding="utf-8-sig")
print("Kruskal-Wallis (osobno dla każdego punktu czasowego):")
display(kw_df_motion)

if dunn_frames_motion:
    dunn_all_motion = pd.concat(dunn_frames_motion)
    dunn_all_motion.to_csv(output_dir / "motion_dunn_posthoc.csv", encoding="utf-8-sig")
    print("\nDunn post-hoc (macierze p skorygowane metodą Holma, osobno dla każdego czasu) zapisano do CSV.")

_time_levels_motion = motion_summary_df["punkt_czasowy_h"].nunique()
_balanced_ids_motion = (
    motion_summary_df.groupby("osobnik_id")["punkt_czasowy_h"].nunique() == _time_levels_motion
)
balanced_ids_motion = _balanced_ids_motion[_balanced_ids_motion].index
balanced_df_motion = motion_summary_df[motion_summary_df["osobnik_id"].isin(balanced_ids_motion)]

friedman_rows_motion = []
for concentration, group in balanced_df_motion.groupby("stężenie_mg"):
    pivoted = group.pivot_table(index="osobnik_id", columns="punkt_czasowy_h", values=VALUE_COL).dropna()
    if pivoted.shape[0] >= 3 and pivoted.shape[1] >= 3:
        friedman_stat, friedman_p = scipy_stats.friedmanchisquare(*[pivoted[c].to_numpy() for c in pivoted.columns])
        friedman_rows_motion.append({
            "stężenie_mg": concentration, "n_osobników": pivoted.shape[0],
            "chi2": friedman_stat, "p_value": friedman_p,
        })
friedman_df_motion = pd.DataFrame(friedman_rows_motion)
friedman_df_motion.to_csv(output_dir / "motion_friedman.csv", index=False, encoding="utf-8-sig")
print("\nFriedman (przebieg czasowy w obrębie każdego stężenia):")
display(friedman_df_motion)